# Práctica de Aprendizaje por Refuerzo: Q-Learning Tabular en FrozenLake-v1

**Estudiante:** Oscar  
**Asignatura:** Inteligencia Artificial y Aprendizaje Profundo / Reinforcement Learning  
**Repositorio GitHub:** [oscar2121/Proyecto-de-Aprendizaje-por-Refuerzo-FrozenLake](https://github.com/oscar2121/Proyecto-de-Aprendizaje-por-Refuerzo-FrozenLake)  

---

## 1. Fundamentos Conceptuales
El Aprendizaje por Refuerzo (RL) modela la interacción entre un **agente** y un **entorno** formalizado como un Proceso de Decisión de Markov (MDP) $(\mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma)$:
- **Estados ($s \in \mathcal{S}$):** 16 celdas en una cuadrícula $4 \times 4$.
- **Acciones ($a \in \mathcal{A}$):** 0: Izquierda, 1: Abajo, 2: Derecha, 3: Arriba.
- **Transición Estocástica (`is_slippery=True`):** La acción seleccionada tiene $P = 1/3$ de éxito y $2/3$ de resbalar lateralmente.
- **Recompensas dispersas:** $r = 1$ en la meta (G), $r = 0$ en hoyos (H) y hielo (F).
- **Ecuación de Bellman (Q-Learning):**
$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

In [ ]:
# Instalación de dependencias para Google Colab si es necesario
!pip install gymnasium numpy matplotlib

import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import random

# Fijar semillas para reproducibilidad
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

## 2. Implementación de la Clase `QLearningAgent`
Implementamos el agente con selección $\varepsilon$-greedy, regla de actualización de diferencia temporal y decaimiento multiplicativo.

In [ ]:
class QLearningAgent:
    def __init__(self, n_states, n_actions, learning_rate=0.8, gamma=0.95,
                 epsilon_start=1.0, epsilon_min=0.01, epsilon_decay=0.9995):
        self.n_states = n_states
        self.n_actions = n_actions
        self.lr = learning_rate
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.q_table = np.zeros((n_states, n_actions))

    def get_action(self, state, training=True):
        """Selección epsilon-greedy: exploración vs explotación"""
        if training and np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        return int(np.argmax(self.q_table[state]))

    def learn(self, state, action, reward, next_state):
        """Actualización TD(0) de Bellman"""
        old_val = self.q_table[state, action]
        next_max = np.max(self.q_table[next_state])
        td_target = reward + self.gamma * next_max
        td_error = td_target - old_val
        self.q_table[state, action] = old_val + self.lr * td_error

    def decay_epsilon(self):
        """Decaimiento exponencial de la tasa de exploración"""
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

## 3. Entrenamiento del Agente (10,000 Episodios)

In [ ]:
env = gym.make('FrozenLake-v1', is_slippery=True)

EPISODES = 10000
MAX_STEPS = 100
agent = QLearningAgent(
    n_states=env.observation_space.n,
    n_actions=env.action_space.n,
    learning_rate=0.8,
    gamma=0.95,
    epsilon_start=1.0,
    epsilon_min=0.01,
    epsilon_decay=0.9995
)

rewards_history = []
epsilon_history = []

print('Iniciando entrenamiento...')
for ep in range(EPISODES):
    state, _ = env.reset()
    total_reward = 0
    
    for _ in range(MAX_STEPS):
        action = agent.get_action(state, training=True)
        next_state, reward, terminated, truncated, _ = env.step(action)
        agent.learn(state, action, reward, next_state)
        state = next_state
        total_reward += reward
        if terminated or truncated:
            break
            
    agent.decay_epsilon()
    rewards_history.append(total_reward)
    epsilon_history.append(agent.epsilon)
    
    if (ep + 1) % 2000 == 0:
        recent_rate = np.mean(rewards_history[-100:]) * 100
        print(f'Episodio {ep+1}/{EPISODES} | Epsilon: {agent.epsilon:.4f} | WinRate ultimos 100: {recent_rate:.1f}%')

print('¡Entrenamiento completado!')

## 4. Visualización de Curvas de Convergencia

In [ ]:
# Curva de media móvil (recompensas acumuladas)
w_size = 100
moving_avg = np.convolve(rewards_history, np.ones(w_size)/w_size, mode='valid')

plt.figure(figsize=(10, 4), dpi=150)
plt.plot(moving_avg, color='black', label=f'Media móvil ({w_size} episodios)')
plt.title('Progreso del Aprendizaje (Convergencia en FrozenLake-v1)', fontweight='bold', color='black')
plt.xlabel('Episodios')
plt.ylabel('Tasa de Éxito')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.show()

## 5. Evaluación Determinista de la Política Final (ε = 0)

In [ ]:
eval_episodes = 1000
eval_rewards = []
eval_steps = []

for ep in range(eval_episodes):
    state, _ = env.reset()
    tot_r = 0
    st = 0
    for _ in range(MAX_STEPS):
        action = agent.get_action(state, training=False)  # Explotación pura
        state, reward, terminated, truncated, _ = env.step(action)
        tot_r += reward
        st += 1
        if terminated or truncated:
            break
    eval_rewards.append(tot_r)
    eval_steps.append(st)

print(f'Tasa de éxito final (Win Rate): {np.mean(eval_rewards)*100:.2f}%')
print(f'Pasos promedio por episodio: {np.mean(eval_steps):.2f}')

## 6. Diagnóstico de la Tabla Q y Política Óptima Resultante

In [ ]:
v_values = np.max(agent.q_table, axis=1).reshape((4, 4))
action_arrows = {0: '←', 1: '↓', 2: '→', 3: '↑'}

print('Valores máximos V(s):')
print(np.round(v_values, 3))

print('\nAcciones óptimas por celda:')
for r in range(4):
    row_str = ''
    for c in range(4):
        idx = r * 4 + c
        if idx in [5, 7, 11, 12]:
            row_str += '  [HOYO] '
        elif idx == 15:
            row_str += '  [META] '
        else:
            best_a = int(np.argmax(agent.q_table[idx]))
            row_str += f'    {action_arrows[best_a]}    '
    print(row_str)